# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and adheres to the FAIR principles, providing structured information on protein abundance, modifications, and sequences in human samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Review all available record sets and their fields by their `@id`.

*If you are using this notebook for the first time, record sets may be large and result in long output. For the full list of available record sets, fields, and columns, refer to the Croissant schema or use the methods below.*

In [ ]:
# List all available record sets and their structure
print("Available record set @id values:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    if 'field' in record_set:
        print("  Fields:")
        fields = record_set['field']
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif isinstance(f, str):
                print(f"    - {f}")
    else:
        print("  (No fields)")
    print()

# Show sample records from each record set
for record_set in dataset.record_sets:
    record_set_id = record_set['@id']
    print(f"Sample rows from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        for i, record in enumerate(records[:2]):
            print(json.dumps(record, indent=2))
        if len(records) > 2:
            print("...\n")
    except Exception as e:
        print(f"  Could not load records: {e}\n")

## 3. Data Extraction
Extract data from a specific record set into a DataFrame. All record set, field, and column names are referenced by their `@id`.

> **Note:** The following example uses the main protein record set (change the `main_record_set_id` variable if required to explore a different data table).

In [ ]:
# You may wish to extract multiple record sets if available.
# Replace with appropriate record set IDs as printed in the previous cell.

main_record_set_id = None
record_set_ids = []
# Identify main record set by description/fields
for record_set in dataset.record_sets:
    rs_id = record_set['@id']
    # Example: choose the one with 'protein' in id, or just pick first for illustration
    fields = record_set.get('field', [])
    # Save all record_set @id for later
    record_set_ids.append(rs_id)
    if not main_record_set_id:
        main_record_set_id = rs_id

# Load all chosen record sets into DataFrames
dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for: {rsid}")
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"{rsid}: Columns: {df.columns.tolist()}")
    print(f"{rsid}: Sample rows:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering by values, normalizing numeric fields, and simple groupby. Field selection is done by `@id`.

Let's pick a numeric field. (Update `numeric_field_id` and `group_field_id` based on the output from above.)

In [ ]:
# Update the following with actual field @id values from your record set
record_set_id = main_record_set_id
df = dataframes[record_set_id]

print("Choose the appropriate numeric field for analysis based on columns listed above.")
# Example: suppose '@id': 'cr:field/coverage_pct' is a numeric field
numeric_field = None
for col in df.columns:
    if 'coverage' in col or 'MW' in col or 'abundance' in col or 'count' in col:
        numeric_field = col
        break
if numeric_field is None and len(df.columns) > 0:
    numeric_field = df.columns[0]  # fallback
print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
try:
    filtered_df = df[df[numeric_field] > threshold]
except Exception:
    filtered_df = df
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field
group_field = None
for col in df.columns:
    if 'sample' in col or 'category' in col or 'group' in col or 'modification' in col:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and compare across categories if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group_field (if found)
if group_field and group_field in df.columns:
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We loaded dataset metadata and inspected available record sets, fields, and columns using their `@id` identifiers (per Croissant schema best practice).
- We extracted tabular data and performed simple exploratory analysis such as filtering, normalization, and grouping.
- Data visualization highlighted basic distributions (details will depend on the specific fields in this dataset).

You can now proceed to more advanced analyses and modeling tasks based on this FAIR-covered mass spectrometry dataset!